In [1]:
# Importa las librerías necesarias
from dotenv import load_dotenv
import openai
import os
from langchain_core.tools import tool
from extract_to_cmaps import prompt_extract_cmapss, generate_cmapss_assistant_prompt
import json
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage 

# Carga las variables de entorno desde el archivo .env
load_dotenv(dotenv_path=".env", override=True)

# Recupera la clave API de OpenAI desde las variables de entorno
openai.api_key = os.getenv("OPENAI_API_KEY")

# Verifica que la clave API se haya cargado correctamente
if openai.api_key is None:
    raise ValueError("La clave API de OpenAI no está configurada correctamente.")

# Ahora puedes usar la clave API en LangChain o directamente con OpenAI
llm = ChatOpenAI(temperature=0, model="gpt-3.5-turbo")  # O el modelo que prefieras


c:\Users\David\anaconda3\envs\ml_python\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Extract to CMAPSS

In [6]:
#@tool
def extract_cmapss_data(message):
    """
    Esta herramienta extrae los datos estructurados para alimentar un modelo de predicción RUL basado en CMAPSS.
    """
    print("\nRecibiendo mensaje:", message)  # Verifica que el mensaje llega
    prompt = prompt_extract_cmapss(message)
    print("\nPrompt generado:", prompt)  # Verifica que el prompt se genera correctamente
    
    # Llamada al modelo
    response = llm(prompt)
    print("\nRespuesta del modelo:", response)  # Verifica lo que devuelve el modelo
    
    try:
        # Intentamos parsear la respuesta para asegurar que está en formato JSON
        parsed_response = json.loads(response)
    except json.JSONDecodeError:
        # Si la respuesta no es un JSON válido, devolver un mensaje de error
        parsed_response = {
            "error": "La respuesta del modelo no es un JSON válido",
            "modelo_seleccionado": "FD001"  # Selección por defecto
        }
    return json.dumps(parsed_response)


In [3]:
# Agrupar las herramientas en una lista
tools = [extract_cmapss_data]

# Vincular las herramientas al modelo OpenAI
llm_with_tools = llm.bind_tools(tools)

print("Herramientas vinculadas correctamente:", llm_with_tools)


Herramientas vinculadas correctamente: bound=ChatOpenAI(profile={'max_input_tokens': 16385, 'max_output_tokens': 4096, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': False, 'structured_output': False, 'image_url_inputs': False, 'pdf_inputs': False, 'pdf_tool_message': False, 'image_tool_message': False, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x0000026795461780>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x00000267BF2CD2A0>, root_client=<openai.OpenAI object at 0x0000026795461210>, root_async_client=<openai.AsyncOpenAI object at 0x00000267954611E0>, temperature=0.0, model_kwargs={}, openai_api_key=SecretStr('**********'), stream_usage=True) kwargs={'tools': [{'type': 'function', 'function': {'name': 'extract_cmapss_data', 'description': 'Esta herrami

In [4]:
# Definir la función del asistente que utiliza las herramientas
def cmapss_assistant(state, config): 
    memory = state.get("loaded_memory", "None")
    
    # Generar el prompt usando la memoria del usuario (si está disponible)
    cmapss_assistant_prompt = generate_cmapss_assistant_prompt(memory)
    
    print("Estado antes de invocar:", state)
    print("Mensajes antes de invocar:", state.get("messages", []))


    # Llamar al modelo con el prompt generado y las herramientas asociadas
    # Ajustar cómo llamas al modelo, asegurando que la herramienta se ejecute
    response = llm_with_tools.invoke([SystemMessage(cmapss_assistant_prompt)] + state["messages"])
    print("Respuesta del modelo después de la invocación:", response)

    # Actualizar el estado con la respuesta del modelo
    return {"messages": [response]}

In [ ]:
from langchain_core.runnables import RunnableConfig

# Simula un mensaje de prueba
mensaje_del_usuario = """
El motor tiene el identificador 200. El ciclo operativo actual es 150. 
Las configuraciones operativas son: setting_1=1, setting_2=2, setting_3=3. 
Los valores de los sensores son: sensor 1: 0.25, sensor 2: 0.35, sensor 3: 0.45, sensor 7: 550.
"""

state = {
    "loaded_memory": "El usuario está interesado en extraer datos de motores aeronáuticos para predicciones RUL.",
    "messages": [HumanMessage(content=mensaje_del_usuario)]  # Asegúrate de que el mensaje esté correctamente formateado
}

# Llamamos a la función cmapss_assistant
response = cmapss_assistant(state, config=RunnableConfig)

# Imprimir la respuesta del asistente
print("Respuesta del asistente:", response)


Estado antes de invocar: {'loaded_memory': 'El usuario está interesado en extraer datos de motores aeronáuticos para predicciones RUL.', 'messages': [HumanMessage(content='\nEl motor tiene el identificador 200. El ciclo operativo actual es 150. \nLas configuraciones operativas son: setting_1=1, setting_2=2, setting_3=3. \nLos valores de los sensores son: sensor 1: 0.25, sensor 2: 0.35, sensor 3: 0.45, sensor 7: 550.\n', additional_kwargs={}, response_metadata={})]}
Mensajes antes de invocar: [HumanMessage(content='\nEl motor tiene el identificador 200. El ciclo operativo actual es 150. \nLas configuraciones operativas son: setting_1=1, setting_2=2, setting_3=3. \nLos valores de los sensores son: sensor 1: 0.25, sensor 2: 0.35, sensor 3: 0.45, sensor 7: 550.\n', additional_kwargs={}, response_metadata={})]
Respuesta del modelo después de la invocación: content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 79, 'prompt_tokens': 309, 'total_

In [7]:
extract_cmapss_data(mensaje_del_usuario)


Recibiendo mensaje: 
El motor tiene el identificador 200. El ciclo operativo actual es 150. 
Las configuraciones operativas son: setting_1=1, setting_2=2, setting_3=3. 
Los valores de los sensores son: sensor 1: 0.25, sensor 2: 0.35, sensor 3: 0.45, sensor 7: 550.

Generated Prompt: 
   Eres un asistente especializado en extraer datos estructurados para alimentar un modelo de predicción RUL basado en CMAPSS.

   TU TAREA:
   Extraer únicamente la información explícita mencionada por el usuario sobre el estado actual de un motor aeronáutico.

   NO DEBES inventar valores.  
   NO estimes sensores no mencionados.  
   NO rellenes medias ni interpolaciones: eso lo hará el modelo después.

   ------------------------------------------------------------
   DATOS QUE DEBES EXTRAER
   ------------------------------------------------------------

   1. unidad  
      - Identificador del motor (si no se menciona → 000)

   2. tiempo_ciclos  
      - Ciclo operativo actual (si no se menciona → 

TypeError: 'ChatOpenAI' object is not callable